In [ ]:
import torch.nn as nn

In [3]:
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        
    def forward(self, x):
        _, (hidden, _) = self.lstm(x)
        return hidden

In [ ]:
class Decoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers):
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, input_dim)
        
    def forward(self, x):

In [ ]:
class LSTMAutoencoder(nn.Module):
    def __init__(
        self,
        input_dim: int = 1,
        hidden_dim: int = 64,
        latent_dim: int = 32,
        num_layers: int = 1,
    ):
        
        super().__init__()
        
        # ---- Encoder ---- #
        self.encoder = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        
        # ---- Latent ---- #
        self.to_latent = nn.Linear(hidden_dim, latent_dim)
        self.from_latent = nn.Linear(latent_dim, hidden_dim)
        
        # ---- Decoder ---- #
        self.decoder = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        
    # ---- Forward ---- #
    def forward(self, x):
        _, (hidden, _) = self.encoder(x)
        hidden = hidden[-1]
        latent = self.to_latent(hidden)
        hidden_dec = self.from_latent(latent)
        seq_len = x.size(1)
        repeated_hidden = hidden_dec.unsqueeze(1).repeat(1, seq_len, 1)
        reconstructed, _ = self.decoder(repeated_hidden)
        return reconstructed